# 03 — Phase 3: Selecting the Arbitration Heads

Per-head **paired** Wilcoxon signed-rank against zero, Benjamini–Hochberg FDR across all 144 heads
within category, and an effect-size floor.

Paired, not unpaired: the design produces token-aligned pairs, and an unpaired test discards
exactly the pairing the design exists to create.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
%load_ext autoreload
%autoreload 2

In [2]:
from circuit_conflict.utils import load_model
from circuit_conflict import dataset as D, metrics as M, pipeline as PL
import numpy as np, json

effects = {c: np.load(PL.RESULTS / "phase1" / f"effects_{c}.npy") for c in ["A", "B", "C"]}
stats_by_cat, head_sets, topk_sets = {}, {}, {}

for c, e in effects.items():
    s = M.paired_head_stats(e); s.insert(0, "category", c)
    s.to_csv(PL.RESULTS / "phase3" / f"head_stats_{c}.csv", index=False)
    stats_by_cat[c] = s
    head_sets[c] = M.select_head_set(s, effect_floor=0.05)
    topk_sets[c] = M.top_k_head_set(s, k=10)
    print(f"  {c}: {len(head_sets[c])} FDR-significant, top-10 selected")

  A: 8 FDR-significant, top-10 selected
  B: 7 FDR-significant, top-10 selected
  C: 7 FDR-significant, top-10 selected


## The selected head sets

In [3]:
for c, s in topk_sets.items():
    print(f"  {c}: " + " ".join(f"L{l}H{h}" for l, h in sorted(s)))

core = topk_sets["A"] & topk_sets["B"] & topk_sets["C"]
print("\nShared by all three:", " ".join(f"L{l}H{h}" for l, h in sorted(core)) or "(none)")

  A: L9H4 L9H9 L10H0 L10H2 L10H6 L10H10 L11H1 L11H2 L11H3 L11H10
  B: L5H1 L8H3 L8H10 L10H0 L10H1 L10H7 L10H11 L11H3 L11H10 L11H11
  C: L0H4 L8H10 L8H11 L9H8 L9H9 L10H0 L10H7 L10H10 L11H2 L11H10

Shared by all three: L10H0 L11H10


## Validation anchor — Ortu et al. (2024)

Category C reproduces their design, so its head set should recover the heads they report for
GPT-2 Small. **If it does not, the pipeline is wrong** — this is the primary end-to-end check on
the whole apparatus.

In [4]:
ORTU = {(9,6),(9,9),(10,0),(10,10),(10,7),(11,10)}
hit = topk_sets["C"] & ORTU
print(f"recovered {len(hit)}/6: " + " ".join(f"L{l}H{h}" for l, h in sorted(hit)))

recovered 5/6: L9H9 L10H0 L10H7 L10H10 L11H10


## Causal confirmation by mean-ablation

Mean-ablation over the control distribution, not zero-ablation: a zeroed head is not a state the
model ever occupies, so zero-ablation conflates removing a head's function with feeding the
residual stream something corrupt.

In [5]:
model = load_model()
admitted = D.load_prompts().query("passes_precondition")
abl = PL.run_ablation(model, admitted, topk_sets)
abl.to_csv(PL.RESULTS / "phase3" / "ablation_results.csv", index=False)

json.dump({c: sorted(map(list, s)) for c, s in topk_sets.items()},
          open(PL.RESULTS / "phase3" / "head_sets_topk.json", "w"), indent=2)
abl.groupby("category").mean_delta.agg(["min", "max"]).round(3)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer
Loaded gpt2 on mps
  n_layers=12, n_heads=12, d_model=768, d_head=64


,min,max
category,,
A,-0.273,0.242
B,-0.447,0.479
C,-3.633,2.483
